# 🔗 MinIO S3 Data Explorer

**Mục đích**: Kết nối MinIO, tải parquet từ data lake và hiển thị nội dung

Notebook này cung cấp một môi trường interactive để:
- Kết nối với MinIO S3
- Liệt kê buckets và files
- Tải dữ liệu Parquet
- Hiển thị dữ liệu chi tiết

* Note: Cần bật docker registry & kafka để xem dữ liệu chi tiết. 

In [1]:
# Cài đặt các thư viện cần thiết
import subprocess
import sys

# Tất cả packages cần thiết cho notebook
packages = [
    'minio',
    'pandas',
    'pyarrow',
    'confluent-kafka',
    'fastavro',
    'cachetools',
    'httpx',
    'attrs',
    'authlib',
    'requests'
]

print("[INFO] Cài đặt thư viện...\n")
failed_packages = []

for package in packages:
    try:
        # Thử import package (handle package name khác import name)
        if package == 'confluent-kafka':
            __import__('confluent_kafka')
        elif package == 'pyarrow':
            __import__('pyarrow')
        elif package == 'httpx':
            __import__('httpx')
        elif package == 'cachetools':
            __import__('cachetools')
        else:
            __import__(package)
        print(f"[OK] {package:<25} - đã có sẵn")
    except ImportError:
        print(f"[INSTALL] Đang cài {package:<25}...")
        try:
            subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", package])
            print(f"[OK] {package:<25} - cài xong")
        except Exception as err:
            print(f"[ERROR] {package:<25} - LỖI: {err}")
            failed_packages.append(package)

if failed_packages:
    print(f"\n[WARN] Các package cài đặt thất bại: {failed_packages}")
else:
    print("\n[OK] Tất cả thư viện sẵn sàng!")

[INFO] Cài đặt thư viện...

[OK] minio                     - đã có sẵn
[OK] pandas                    - đã có sẵn
[OK] pyarrow                   - đã có sẵn
[OK] confluent-kafka           - đã có sẵn
[OK] fastavro                  - đã có sẵn
[OK] cachetools                - đã có sẵn
[OK] httpx                     - đã có sẵn
[OK] attrs                     - đã có sẵn
[OK] authlib                   - đã có sẵn
[OK] requests                  - đã có sẵn

[OK] Tất cả thư viện sẵn sàng!


In [2]:
# Import thư viện
import os
from minio import Minio
from minio.error import S3Error
import pandas as pd
import io

# Cấu hình kết nối MinIO
MINIO_CONFIG = {
    'endpoint': os.getenv('MINIO_ENDPOINT', 'localhost:9000'),
    'access_key': os.getenv('MINIO_ACCESS_KEY', 'minioadmin'),
    'secret_key': os.getenv('MINIO_SECRET_KEY', 'minioadmin'),
    'region': os.getenv('MINIO_REGION', 'ap-southeast-1'),
    'use_ssl': False
}

print(f"[INFO] Kết nối MinIO tại: {MINIO_CONFIG['endpoint']}")

try:
    # Khởi tạo client MinIO
    client = Minio(
        endpoint=MINIO_CONFIG['endpoint'],
        access_key=MINIO_CONFIG['access_key'],
        secret_key=MINIO_CONFIG['secret_key'],
        region=MINIO_CONFIG['region'],
        secure=MINIO_CONFIG['use_ssl']
    )

    print("[OK] Kết nối MinIO thành công!")

except Exception as err:
    print(f"[ERROR] Lỗi kết nối: {err}")
    print(f"[HINT] Đảm bảo MinIO đang chạy tại {MINIO_CONFIG['endpoint']}")

[INFO] Kết nối MinIO tại: localhost:9000
[OK] Kết nối MinIO thành công!


In [3]:
# Liệt kê tất cả buckets
print("[INFO] Danh sách Buckets:\n")

try:
    buckets = list(client.list_buckets())

    if buckets:
        for i, bucket in enumerate(buckets, 1):
            print(f"{i}. {bucket.name}")
        print(f"\n[OK] Tổng cộng: {len(buckets)} bucket")
    else:
        print("[WARN] Không có bucket nào. Tạo 'data-lake' bucket...")
        client.make_bucket('data-lake')
        print("[OK] Bucket 'data-lake' đã tạo")

except Exception as err:
    print(f"[ERROR] Lỗi: {err}")

[INFO] Danh sách Buckets:

1. data-lake

[OK] Tổng cộng: 1 bucket


In [11]:
# Tìm và tải file parquet
print("[INFO] Tìm file Parquet...\n")

bucket_name = 'data-lake'
parquet_files = []

try:
    # Tìm tất cả file .parquet
    for obj in client.list_objects(bucket_name, recursive=True):
        if obj.object_name.endswith('.parquet'):
            parquet_files.append(obj.object_name)

    if parquet_files:
        print(f"[OK] Tìm thấy {len(parquet_files)} file Parquet:\n")
        for i, file in enumerate(parquet_files[:5], 1):
            print(f"{i}. {file}")

        if len(parquet_files) > 5:
            print(f"... và {len(parquet_files) - 5} file khác")
    else:
        print("[WARN] Chưa có file Parquet. Hãy chạy Spark job để tạo dữ liệu.")

except S3Error as err:
    print(f"[ERROR] Lỗi: {err}")

[INFO] Tìm file Parquet...

[OK] Tìm thấy 7 file Parquet:

1. warehouse/bronze/bronze_delivery_status/data/event_time_day=2026-04-13/00175-675-691905b0-82da-4553-8d1c-6813c1964c17-0-00001.parquet
2. warehouse/bronze/bronze_orders/data/event_time_day=2026-04-13/00175-240-079dd343-549a-4e5a-a1ba-8ddce4541128-0-00001.parquet
3. warehouse/bronze/bronze_payments/data/event_time_day=2026-04-13/00175-1328-cfaf7add-e56a-420b-8aee-5cccda1dd330-1-00001.parquet
4. warehouse/bronze/bronze_payments/data/event_time_day=2026-04-13/00175-22-cfaf7add-e56a-420b-8aee-5cccda1dd330-0-00001.parquet
5. warehouse/bronze/bronze_products/data/event_time_day=2026-04-13/00175-894-be5d97a7-6a63-4831-8a70-eb7b0cb54ad2-0-00001.parquet
... và 2 file khác


In [3]:
# Topic decoder (concise + fast)
import io
import json
import os
from io import BytesIO

import fastavro
import pandas as pd
import requests
from minio import Minio

SCHEMA_REGISTRY_URL = "http://localhost:8081"
SCHEMA_REGISTRY_TIMEOUT_SEC = 1.5
schema_cache = {}
parquet_index = {}

BRONZE_TABLES = {
    "orders": "bronze_orders",
    "payments": "bronze_payments",
    "shipments": "bronze_shipments",
    "delivery": "bronze_delivery_status",
    "users": "bronze_users",
    "products": "bronze_products",
}


def _ensure_minio_context(refresh: bool = False):
    global client, bucket_name, parquet_files, parquet_index

    bucket_name = globals().get("bucket_name", "data-lake")

    if "client" not in globals() or client is None:
        client = Minio(
            endpoint=os.getenv("MINIO_ENDPOINT", "localhost:9000"),
            access_key=os.getenv("MINIO_ACCESS_KEY", "minioadmin"),
            secret_key=os.getenv("MINIO_SECRET_KEY", "minioadmin"),
            region=os.getenv("MINIO_REGION", "ap-southeast-1"),
            secure=False,
        )

    if "parquet_files" not in globals():
        parquet_files = []

    need_reload = refresh or len(parquet_files) == 0
    if need_reload:
        parquet_files = [
            obj.object_name
            for obj in client.list_objects(bucket_name, recursive=True)
            if obj.object_name.endswith(".parquet")
        ]

    need_reindex = refresh or (not isinstance(parquet_index, dict)) or len(parquet_index) == 0
    if need_reload or need_reindex:
        parquet_index = {
            topic: sorted([p for p in parquet_files if table in p.lower()])
            for topic, table in BRONZE_TABLES.items()
        }


def get_schema_from_registry(schema_id: int, registry_url: str = SCHEMA_REGISTRY_URL):
    if schema_id in schema_cache:
        return schema_cache[schema_id]

    try:
        response = requests.get(
            f"{registry_url}/schemas/ids/{schema_id}",
            timeout=SCHEMA_REGISTRY_TIMEOUT_SEC,
        )
        if response.status_code == 200:
            schema = json.loads(response.json().get("schema", "{}"))
            schema_cache[schema_id] = schema
            return schema
    except Exception:
        pass

    schema_cache[schema_id] = None
    return None


def deserialize_kafka_avro(raw_bytes, schema_id_hint=None):
    """Support both formats:
    1) Confluent wire format: [0x00][schema_id(4)][payload]
    2) Raw Avro payload + schema_id column
    """
    try:
        if not isinstance(raw_bytes, (bytes, bytearray)):
            return None
        if len(raw_bytes) == 0:
            return None

        schema_id = None
        payload = raw_bytes

        # Confluent wire format
        if len(raw_bytes) >= 6 and raw_bytes[0:1] == b"\x00":
            schema_id = int.from_bytes(raw_bytes[1:5], "big")
            payload = raw_bytes[5:]
        elif schema_id_hint is not None:
            # Raw payload in Bronze, schema_id stored separately
            try:
                schema_id = int(schema_id_hint)
            except Exception:
                schema_id = None

        if schema_id is None:
            return None

        schema = get_schema_from_registry(schema_id)
        if not schema:
            return None

        return fastavro.schemaless_reader(BytesIO(payload), schema)
    except Exception:
        return None


def view_topic(topic_name: str, limit: int = 10, show_raw_bytes: bool = False, refresh_index: bool = False):
    if topic_name not in BRONZE_TABLES:
        print(f"[ERROR] Topic '{topic_name}' not found. Available: {', '.join(BRONZE_TABLES)}")
        return None

    try:
        _ensure_minio_context(refresh=refresh_index)
    except Exception as e:
        print(f"[ERROR] MinIO context error: {e}")
        return None

    matching_files = parquet_index.get(topic_name, [])
    if not matching_files:
        print(f"[ERROR] No files for topic '{topic_name}'")
        return None

    file_to_load = matching_files[-1]

    response = None
    try:
        response = client.get_object(bucket_name, file_to_load)
        df = pd.read_parquet(io.BytesIO(response.read()))

        total_records = len(df)
        sample_df = df.head(min(limit, total_records)).copy()

        if "raw_value" in sample_df.columns:
            rows = []
            for _, row in sample_df.iterrows():
                record = {
                    "event_source": row.get("event_source", "?"),
                    "event_time": row.get("event_time", "?"),
                }
                decoded = deserialize_kafka_avro(row["raw_value"], row.get("schema_id"))
                if isinstance(decoded, dict):
                    record.update(decoded)
                if show_raw_bytes:
                    raw = row.get("raw_value", b"")
                    record["raw_value_len"] = len(raw) if isinstance(raw, (bytes, bytearray)) else 0
                rows.append(record)
            out_df = pd.DataFrame(rows)
        else:
            out_df = sample_df

        print(f"[INFO] {topic_name.upper()} | file: {file_to_load.split('/')[-1]}")
        print(f"Total in file: {total_records:,} | showing: {len(out_df)}")

        if not out_df.empty:
            print("-" * 100)
            with pd.option_context("display.max_colwidth", 80, "display.width", 200):
                print(out_df.to_string(index=False))
        else:
            print("(No rows to display)")

        return out_df

    except Exception as e:
        print(f"[ERROR] {e}")
        return None

    finally:
        if response is not None:
            response.close()
            response.release_conn()


print("[OK] Ready. Use view_topic('users', 3) or show_samples('users', 3)")

[OK] Ready. Use view_topic('users', 3) or show_samples('users', 3)


In [2]:
# SIMPLE FUNCTION: Select Topic & Show K Samples

def show_samples(topic: str, k: int = 5):
    """
    Đơn giản: Chọn topic và in ra K mẫu

    Args:
        topic: Tên topic ('orders', 'payments', 'shipments', 'delivery', 'users', 'products')
        k: Số mẫu cần hiển thị

    Examples:
        show_samples('orders', 3)       # In 3 orders
        show_samples('users', 5)        # In 5 user changes
        show_samples('payments', 10)    # In 10 payments
    """

    print(f"\n{'='*100}")
    print(f"[INFO] Showing {k} samples from '{topic.upper()}' topic")
    print(f"{'='*100}\n")

    return view_topic(topic, limit=k)


# Available topics
TOPICS_INFO = {
    'orders': {
        'description': 'Customer Orders (Streaming)',
        'fields': ['order_id', 'user_id', 'product_id', 'amount', 'currency', 'ts'],
        'count': 'All orders placed'
    },
    'payments': {
        'description': 'Payment Events (Streaming)',
        'fields': ['payment_id', 'order_id', 'method', 'status', 'amount', 'ts'],
        'count': 'All payment transactions'
    },
    'shipments': {
        'description': 'Shipment Events (Streaming)',
        'fields': ['shipment_id', 'order_id', 'eta_days', 'carrier', 'ts'],
        'count': 'All shipments'
    },
    'delivery': {
        'description': 'Delivery Status Updates (Streaming)',
        'fields': ['delivery_id', 'shipment_id', 'status', 'reason', 'ts'],
        'count': 'DELIVERED/FAILED/RETURNED'
    },
    'users': {
        'description': 'User Changes via CDC (Debezium)',
        'fields': ['user_id', 'name', 'email', 'country', 'op', 'ts_ms'],
        'count': 'before/after with operation'
    },
    'products': {
        'description': 'Product Changes via CDC (Debezium)',
        'fields': ['product_id', 'name', 'category', 'price', 'supplier_id', 'op', 'ts_ms'],
        'count': 'before/after with operation'
    }
}

print("\n" + "="*100)
print("[INFO] TOPIC SELECTOR & SAMPLE VIEWER")
print("="*100 + "\n")

print("Available topics:\n")
for i, (topic, info) in enumerate(TOPICS_INFO.items(), 1):
    print(f"{i}. {topic.upper():<12} - {info['description']}")
    print(f"   Fields: {', '.join(info['fields'][:3])}...")
    print()

print("\nUsage:")
print("  show_samples('orders', 5)       # Show 5 orders")
print("  show_samples('users', 3)        # Show 3 user changes")
print("  show_samples('payments', 10)    # Show 10 payments")
print("  show_samples('delivery', 2)     # Show 2 delivery status")
print()

print("Or call view_topic for more details:")
print("  view_topic('orders', limit=5)   # Full detailed view")


[INFO] TOPIC SELECTOR & SAMPLE VIEWER

Available topics:

1. ORDERS       - Customer Orders (Streaming)
   Fields: order_id, user_id, product_id...

2. PAYMENTS     - Payment Events (Streaming)
   Fields: payment_id, order_id, method...

3. SHIPMENTS    - Shipment Events (Streaming)
   Fields: shipment_id, order_id, eta_days...

4. DELIVERY     - Delivery Status Updates (Streaming)
   Fields: delivery_id, shipment_id, status...

5. USERS        - User Changes via CDC (Debezium)
   Fields: user_id, name, email...

6. PRODUCTS     - Product Changes via CDC (Debezium)
   Fields: product_id, name, category...


Usage:
  show_samples('orders', 5)       # Show 5 orders
  show_samples('users', 3)        # Show 3 user changes
  show_samples('payments', 10)    # Show 10 payments
  show_samples('delivery', 2)     # Show 2 delivery status

Or call view_topic for more details:
  view_topic('orders', limit=5)   # Full detailed view


In [4]:
# DEMO: Using show_samples to view topics

print("\n" + "="*100)
print("[INFO] DEMO: How to use show_samples()")
print("="*100 + "\n")

print("Example 1: Show 5 users (CDC changes)")
print("-" * 100)
print("Code: show_samples('users', 5)\n")

# df_products_5 = show_samples('products', 5)
df_orders_5 = show_samples('orders', 5)


[INFO] DEMO: How to use show_samples()

Example 1: Show 5 users (CDC changes)
----------------------------------------------------------------------------------------------------
Code: show_samples('users', 5)


[INFO] Showing 5 samples from 'ORDERS' topic

[INFO] ORDERS | file: 00175-9395-079dd343-549a-4e5a-a1ba-8ddce4541128-7-00001.parquet
Total in file: 1,207 | showing: 5
----------------------------------------------------------------------------------------------------
event_source                       event_time
   orders.v1 2026-04-13 03:29:32.864000+00:00
   orders.v1 2026-04-13 03:29:32.889000+00:00
   orders.v1 2026-04-13 03:29:32.894000+00:00
   orders.v1 2026-04-13 03:29:32.917000+00:00
   orders.v1 2026-04-13 03:29:32.969000+00:00
